In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# ⬇️ SB3 + Gym imports
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

# ⬇️ Your imports
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state


In [3]:

# ⬇️ Custom wrapper to convert dict obs → flat array
class FlattenedBlockEnv(gym.Env):
    def __init__(self, width=12, height=12, num_blocks=3):
        super().__init__()
        self.raw_env = BlockPuzzleEnv(width, height, num_blocks)
        self.action_space = self.raw_env.action_space
        dummy_obs, _ = self.raw_env.reset()
        sample_obs = encode_state(dummy_obs)
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=sample_obs.shape, dtype=np.float32
        )
    
    def reset(self, seed=None, options=None):
        obs_dict, _ = self.raw_env.reset()
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), {}

    def step(self, action):
        # Handle batched action from DummyVecEnv (e.g., [0, 9, 10])
        if isinstance(action, (list, np.ndarray)) and len(action) == 3:
            a0, a1, a2 = map(int, action)
        else:
            # Flat index → unravel into (block_index, row, col)
            a0, a1, a2 = np.unravel_index(action, self.action_space.nvec)

        raw_action = (a0, a1, a2)
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(raw_action)
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info


    def render(self):
        return self.raw_env.render()

    def close(self):
        return self.raw_env.close()


class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space

        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))
        self.observation_space = spaces.Box(low=0, high=1, shape=(encode_state(self.raw_env.reset()[0]).shape[0],), dtype=np.float32)

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        return encode_state(obs_dict).astype(np.float32), {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [4]:


class GreedyEvalCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=1000, n_eval_episodes=5, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            original_exploration_rate = self.model.exploration_rate
            self.model.exploration_rate = 0.0  # Greedy mode

            rewards = []
            for _ in range(self.n_eval_episodes):
                obs = self.eval_env.reset()
                done = False
                total_reward = 0
                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, reward, done, info = self.eval_env.step(action)
                    total_reward += reward
                rewards.append(total_reward)

            mean_reward = np.mean(rewards)

            if self.verbose > 0:
                print(f"[Greedy Eval] Step {self.n_calls} - Mean Reward: {mean_reward:.2f}")

            # Log to TensorBoard
            if self.logger is not None:
                self.logger.record('eval/greedy_mean_reward', mean_reward, exclude='stdout')

            self.model.exploration_rate = original_exploration_rate

        return True


In [5]:
class CustomMetricsCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            for key, value in info.items():
                if isinstance(value, np.ndarray):
                    self.logger.record(f"custom/{key}", value.mean())
                else:
                    self.logger.record(f"custom/{key}", value)
        return True

In [6]:
raw_env = BlockPuzzleEnv(width=12, height=12, num_blocks=3)
wrapped_env = DiscreteActionWrapper(raw_env)
monitored_env = Monitor(wrapped_env)

check_env(wrapped_env, warn=True)  # ✅ Now this should pass
vec_env = DummyVecEnv([lambda: monitored_env])

model = PPO(
    policy="MlpPolicy",
    env=vec_env,
    learning_rate=1e-3,
    n_steps=2048,
    batch_size=512,
    gamma=0.99,
    device='cpu',
    verbose=1,
    tensorboard_log="./sb3_logs/"
)

callback = CustomMetricsCallback()
model.learn(total_timesteps=1000000, callback=callback)
model.save("sb3_block_ppo_mlp")


Using cpu device
Logging to ./sb3_logs/PPO_14
------------------------------------------------------------------
| custom/                 |                                      |
|    TimeLimit.truncated  | False                                |
|    episode              | {'r': 2.233, 'l': 293, 't': 1.133... |
|    invalid_moves        | 1957                                 |
|    move_count           | 102                                  |
|    terminal_observation | 0.3783784                            |
| rollout/                |                                      |
|    ep_len_mean          | 562                                  |
|    ep_rew_mean          | 1.33                                 |
| time/                   |                                      |
|    fps                  | 2966                                 |
|    iterations           | 1                                    |
|    time_elapsed         | 0                                    |
|    total_times

KeyboardInterrupt: 

In [ ]:
obs, _ = wrapped_env.reset()
done = False
total_reward = 0
moves = 0

while not done:
    action, _  = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = wrapped_env.step(action)
    total_reward += reward
    done = terminated or truncated
    wrapped_env.render()
    print()

print(f"Total reward (greedy run): {total_reward}")

Total reward (greedy run): 0
